# 05 Neural network: perceptron and MLP

A single neuron (the perceptron) and a small multilayer perceptron (MLP) price the same bonds, against matrix pricing and a random forest fitted on the same rows. The question: does a neural network add anything over tree ensembles on tabular data like ours?

Class notebooks followed: `Perceptron_MLP_IBM_attrition` (the structure) and `Mnist_mlp_CNN_generalization` (dense layers, dropout, early stopping). Changes for our data: a regression, so the output neuron is linear and the loss is MAE instead of a sigmoid and cross-entropy; the early-stopping months are the last three training months instead of a random 20%; batches of 256 rows instead of 32, since we have about 100 times more rows; no cutoff step, which only applies to classifiers.

Validation months only: the test months are scored once, in `99_final`. Runs top to bottom in about 10 minutes on a laptop; figures go to
`reports/figures/05_*.png`.

In [ ]:
# Deck style for every figure: IG is always teal, HY always orange
import matplotlib.pyplot as plt

DECK = {"teal": "#114A6B", "orange": "#FF5F00", "slate": "#3A4550", "grey": "#8A949C", "pale": "#DCE6EC"}
plt.rcParams.update({
    "font.family": "Arial", "font.size": 12,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": DECK["grey"], "axes.labelcolor": DECK["slate"], "axes.titleweight": "bold",
    "axes.titlecolor": DECK["teal"],
    "axes.prop_cycle": plt.cycler(color=[DECK["teal"], DECK["orange"], DECK["grey"], DECK["slate"]]),
    "xtick.color": DECK["slate"], "ytick.color": DECK["slate"], "axes.grid": True, "axes.grid.axis": "y",
    "axes.axisbelow": True, "grid.color": DECK["pale"],
    "figure.figsize": (10, 5.6), "figure.dpi": 100, "savefig.dpi": 200, "savefig.bbox": "tight",
})

## 1. Setup

In [ ]:
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GridSearchCV, cross_val_score, cross_validate
from sklearn.pipeline import Pipeline

from ml2.baseline import matrix_same_date
from ml2.config import load_config, resolve
from ml2.data import check_panel, load_panel
from ml2.evaluate import improvement_ci, metrics_by_stratum, regression_metrics
from ml2.features import make_preprocessor, xy
from ml2.splits import issuer_holdout_folds, walk_forward_folds

cfg = load_config()
TARGET = cfg["problem"]["target"]
PREFIX = "05"
FIG = resolve("reports/figures")
FIG.mkdir(parents=True, exist_ok=True)

def save(fig, name):
    """Save a figure to reports/figures/<prefix>_<name>.png."""
    fig.savefig(FIG / f"{PREFIX}_{name}.png", dpi=150, bbox_inches="tight")


check_panel()
df = load_panel(partitions=["train", "valid"])
train, valid = df[df.partition == "train"], df[df.partition == "valid"]
X_tr, y_tr = xy(train)
X_va, y_va = xy(valid)

# Searches run on a 30% sample of train (time order kept); chosen models are refitted on all of train
sub = train.sample(frac=0.3, random_state=42).sort_values("snapshot_date")
X_s, y_s = xy(sub)
cv = walk_forward_folds(sub.loc[X_s.index, "snapshot_date"])  # replaces cv=5 / KFold in the class notebooks
# New-issuer question: in each time-ordered fold, a fifth of the companies are held out (see ml2.splits)
new_issuer_folds = issuer_holdout_folds(sub.loc[X_s.index, "snapshot_date"], sub.loc[X_s.index, "issuer_ticker"])


def new_issuer_cost(model):
    """MAE on the held-out companies, trained with and without them (same number of rows, same months)."""
    seen, unseen = [], []
    for tr_seen, tr_unseen, held in new_issuer_folds:
        X_h, y_h = X_s.iloc[held], y_s.iloc[held]
        seen.append(mean_absolute_error(y_h, clone(model).fit(X_s.iloc[tr_seen], y_s.iloc[tr_seen]).predict(X_h)))
        unseen.append(mean_absolute_error(
            y_h, clone(model).fit(X_s.iloc[tr_unseen], y_s.iloc[tr_unseen]).predict(X_h)))
    return float(np.mean(seen)), float(np.mean(unseen))
print(f"train {X_tr.shape}, valid {X_va.shape}, search sample {X_s.shape}, {len(cv)} time-ordered folds")

import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import keras
import tensorflow as tf
from keras import layers
from sklearn.ensemble import RandomForestRegressor

SEED = 42
keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()  # same weights on every rerun
print("TensorFlow:", tf.__version__, "| Keras:", keras.__version__)

In [ ]:
# The bar: matrix pricing re-marked on each date (the gate baseline), on valid. About 1.5 minutes.
valid = valid.copy()
valid["pred_matrix_same_date"] = matrix_same_date(valid)
results = {}  # model name -> metrics by grade on valid, for the comparison table


def vs_bar(pred, name):
    """MAE by grade for predictions on X_va's rows, stored in `results`, shown next to the gate baseline."""
    rows = valid.loc[X_va.index].assign(pred=np.asarray(pred))
    results[name] = metrics_by_stratum(rows, TARGET, "pred", "grade_bucket").set_index("stratum")
    bar = metrics_by_stratum(rows, TARGET, "pred_matrix_same_date", "grade_bucket").set_index("stratum")
    out = pd.DataFrame({name: results[name]["mae_bps"], "matrix re-marked (bar)": bar["mae_bps"]}).T
    out["IG gain"] = 1 - out["IG"] / out.loc["matrix re-marked (bar)", "IG"]
    out["HY gain"] = 1 - out["HY"] / out.loc["matrix re-marked (bar)", "HY"]
    return out[["IG", "HY", "ALL", "IG gain", "HY gain"]].round(3)


vs_bar(valid.loc[X_va.index, "pred_matrix_same_date"], "Matrix pricing re-marked each date (bar)")

## 2. Early-stopping months inside train

The class notebook splits 20% of the training rows off at random to decide when to stop training. A random split
leaks here (a bond's spread this month is close to last month's), so the last three training months play that role
instead. Valid is not used for any decision in this notebook until the comparison.

In [ ]:
months = np.sort(train["snapshot_date"].unique())
stop_from = months[-3]
in_stop = (train.loc[X_tr.index, "snapshot_date"] >= stop_from).to_numpy()
fit_idx, stop_idx = X_tr.index[~in_stop], X_tr.index[in_stop]
print(f"fit: {len(fit_idx):,} rows up to {pd.Timestamp(months[-4]).date()}; "
      f"early stopping: {len(stop_idx):,} rows from {pd.Timestamp(stop_from).date()}")

## 3. Preprocessing: what changes versus trees

A tree splits on thresholds, so the scale of a feature does not matter. A neural network multiplies each input by a
weight and adds them up, so `total_debt` in dollars would swamp `coupon_pct`, and gradient descent would be slow and
unstable. `make_preprocessor()` does what the class notebook does: median imputation (with a flag for each missing
value), `StandardScaler`, and one-hot encoding. It is fitted on the fit months only.

The spread is given to the network in percent (bp / 100), so the output starts on roughly the right scale;
predictions are multiplied back to bp.

In [ ]:
prep_nn = make_preprocessor()  # median imputation with missing flags, StandardScaler, one-hot
A_fit = prep_nn.fit_transform(X_tr.loc[fit_idx]).astype("float32")
A_stop = prep_nn.transform(X_tr.loc[stop_idx]).astype("float32")
A_va = prep_nn.transform(X_va).astype("float32")

SCALE = 100.0
t_fit = (y_tr.loc[fit_idx] / SCALE).to_numpy("float32")
t_stop = (y_tr.loc[stop_idx] / SCALE).to_numpy("float32")
n_features = A_fit.shape[1]
print("Input features after encoding:", n_features)

## 4. Helper functions

The loss is the mean absolute error, the metric the gate uses. MAE is also less pulled by the few very wide HY
spreads than squared error. With dropout and L2, the training loss includes the penalty, so the curves plot the MAE
itself, in bp.

In [ ]:
def plot_history(h, title):
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(np.array(h.history["mae"]) * SCALE, label="fit months")
    ax.plot(np.array(h.history["val_mae"]) * SCALE, label="early-stopping months")
    ax.set_title(title); ax.set_xlabel("Epoch"); ax.set_ylabel("MAE (bp)"); ax.legend()
    save(fig, "history_" + title.lower().replace(" ", "_").replace(",", ""))
    plt.show()


def fit_nn(model, lr, patience=10, epochs=100):
    """Adam on MAE, batches of 256 rows, early stopping on the last training months (best weights kept)."""
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="mae", metrics=["mae"])
    es = keras.callbacks.EarlyStopping(monitor="val_mae", patience=patience, restore_best_weights=True)
    return model.fit(A_fit, t_fit, validation_data=(A_stop, t_stop), epochs=epochs, batch_size=256,
                     callbacks=[es], verbose=0)


def predict_bp(model, A):
    return model.predict(A, batch_size=4096, verbose=0).ravel() * SCALE

## 5. The perceptron: one neuron, no hidden layer

One neuron with no activation computes `w . x + b`: a linear regression, fitted by gradient descent (Adam) instead
of a formula. With MAE as the loss it is a median regression. It can only fit a straight-line relationship, so it is
the linear benchmark for everything else.

In [ ]:
keras.utils.set_random_seed(SEED)
perceptron = keras.Sequential([
    layers.Input(shape=(n_features,)),
    layers.Dense(1),  # the single neuron, linear output for a regression
])
h_perc = fit_nn(perceptron, lr=1e-2)
perceptron.summary()
print("Stopped after", len(h_perc.history["loss"]), "epochs")
plot_history(h_perc, "Perceptron")
vs_bar(predict_bp(perceptron, A_va), "Perceptron (one neuron)")

**Reading** (to write from the output above). How far is a straight line from the bar? Count the parameters in the summary: one weight per input plus the bias.

## 6. Multilayer perceptron (MLP)

Hidden layers with ReLU let the network learn non-linear patterns and interactions (a wide spread for a long, volatile,
indebted company together). Overfitting is the main risk, so, as in the class notebook: two hidden layers, dropout,
L2 weight decay and early stopping.

We have about 100 times more rows than the class example (about 100,000 against 1,000), so the network can be
wider and needs less regularization. We try three settings and choose on the early-stopping months, not on valid:
no dropout and no L2, a light setting, and the class notebook's setting.

In [ ]:
def make_mlp(dropout, l2):
    reg = keras.regularizers.l2(l2) if l2 else None
    return keras.Sequential([
        layers.Input(shape=(n_features,)),
        layers.Dense(64, activation="relu", kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(32, activation="relu", kernel_regularizer=reg),
        layers.Dropout(dropout),
        layers.Dense(1),
    ])


settings = {"no dropout, no L2": (0.0, 0.0), "light (dropout 0.1, L2 1e-5)": (0.1, 1e-5),
            "class (dropout 0.3, L2 1e-3)": (0.3, 1e-3)}
mlps, histories, stop_mae = {}, {}, {}
for name, (dropout, l2) in settings.items():
    keras.utils.set_random_seed(SEED)
    t0 = time.perf_counter()
    mlps[name] = make_mlp(dropout, l2)
    histories[name] = fit_nn(mlps[name], lr=1e-3)
    stop_mae[name] = min(histories[name].history["val_mae"]) * SCALE
    print(f"{name}: {len(histories[name].history['loss'])} epochs, early-stopping MAE {stop_mae[name]:.1f} bp, "
          f"{time.perf_counter() - t0:.0f} s")
chosen = min(stop_mae, key=stop_mae.get)
mlp = mlps[chosen]
print("chosen on the early-stopping months:", chosen)
mlp.summary()

In [ ]:
plot_history(histories[chosen], "MLP")
vs_bar(predict_bp(mlp, A_va), f"MLP, {chosen}")

**Reading** (to write from the output above). Where does the early-stopping MAE stop improving while the fit MAE keeps falling? Which setting won, and why do the class settings, made for 1,000 rows, behave differently on 100,000?

## 7. Random forest on the same rows

As in the class notebook, a random forest fitted on the same fit months (no scaling needed), so the comparison is
fair. The tuned forest in `03_trees_ensembles` also saw the last three training months.

In [ ]:
rf = Pipeline([("pre", make_preprocessor(scale=False, one_hot=False)),
               ("rf", RandomForestRegressor(n_estimators=100, min_samples_leaf=10, max_features=0.33,
                                            random_state=SEED, n_jobs=-1))])
t0 = time.perf_counter()
rf.fit(X_tr.loc[fit_idx], y_tr.loc[fit_idx])
print(f"fit {time.perf_counter() - t0:.0f} s")
vs_bar(rf.predict(X_va), "Random forest (same rows)")

**Reading** (to write from the output above). Does the MLP beat the forest? Why are tree ensembles hard to beat on tabular data?

## 8. Predicted against actual

The class notebook's ROC curves and confusion matrices are for classifiers. For a regression, the matching picture
is predicted against actual spread on valid, by grade.

In [ ]:
rows = valid.loc[X_va.index, [TARGET, "grade_bucket"]].assign(mlp=predict_bp(mlp, A_va), rf=rf.predict(X_va))
pick = rows.sample(5000, random_state=SEED)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5), sharex=True, sharey=True)
for ax, col, title in zip(axes, ["mlp", "rf"], ["MLP", "Random forest"]):
    for g, colour in [("IG", DECK["teal"]), ("HY", DECK["orange"])]:
        s = pick[pick.grade_bucket == g]
        ax.scatter(s[TARGET], s[col], s=4, alpha=0.4, color=colour, label=g)
    ax.plot([0, 1000], [0, 1000], color=DECK["slate"], lw=0.8)
    ax.set_xscale("symlog", linthresh=50); ax.set_yscale("symlog", linthresh=50)
    ax.set_title(title); ax.set_xlabel("Actual spread (bp)"); ax.legend()
axes[0].set_ylabel("Predicted spread (bp)")
save(fig, "predicted_vs_actual")
plt.show()

**Reading** (to write from the output above). Where do the two models miss: the widest HY bonds, the tightest IG bonds?

## 9. What the perceptron weights tell us

The perceptron is linear and its inputs are standardized, so each weight is the change in spread (in bp) for one
standard deviation more of that feature, all else equal: the same reading as a regression coefficient. Positive
means a wider spread.

In [ ]:
names = prep_nn.get_feature_names_out()
w = pd.Series(perceptron.layers[0].get_weights()[0].ravel() * SCALE, index=names).sort_values()
top = pd.concat([w.head(8), w.tail(8)])
fig, ax = plt.subplots(figsize=(8, 5.5))
ax.barh(top.index, top.values, color=[DECK["teal"] if v < 0 else DECK["orange"] for v in top])
ax.set_title("Perceptron weights (top 8 each side)"); ax.set_xlabel("bp per standard deviation")
save(fig, "perceptron_weights")
plt.show()

**Reading** (to write from the output above). Do the signs make financial sense (volatility, leverage, maturity, HY)? Compare with the tree importances in `03_trees_ensembles`.

## 10. Comparison table and RUN_LOG

In [ ]:
table = pd.concat({name: r["mae_bps"] for name, r in results.items()}, axis=1).T[["IG", "HY", "ALL"]]
bar = table.loc["Matrix pricing re-marked each date (bar)"]
table["IG gain vs bar"] = 1 - table["IG"] / bar["IG"]
table["HY gain vs bar"] = 1 - table["HY"] / bar["HY"]
extra = pd.DataFrame({name: r.loc["ALL", ["rmse_bps", "r2"]] for name, r in results.items()}).T
table = table.join(extra).round(3)
table.to_csv(FIG.parent / f"{PREFIX}_comparison_valid.csv")
table

### Lines for RUN_LOG.md

For `notebooks/RUN_LOG.md`; the parameters are printed in the last cell.

In [ ]:
today = pd.Timestamp.today().strftime("%Y-%m-%d")
for name, r in table.iterrows():
    print(f"| {today} | <your name> | 05_neural_network | {name} | all 33 | fit on train to Feb 2025, early stopping on Mar to May 2025, scored on valid | "
          f"{r['IG']:.1f} / {r['HY']:.1f} | gain vs bar IG {r['IG gain vs bar']:.1%}, "
          f"HY {r['HY gain vs bar']:.1%} |")

In [ ]:
print("MLP setting:", chosen, "| epochs:", len(histories[chosen].history["loss"]),
      "| perceptron epochs:", len(h_perc.history["loss"]))

## Questions to discuss

1. Does the MLP beat the single perceptron? What does that say about how non-linear spreads are?
2. Does the MLP beat the random forest on the same rows, and the boosted trees in `03`? Why are tree ensembles hard
   to beat on tabular data?
3. Which preprocessing steps were required for the networks but not for the forest?
4. What happened without dropout and L2? Why do the class settings fit our data differently?
5. Would a pricing desk run this network every day? Think of training time, reruns giving different weights, and
   explaining a price to an auditor.